**0. CẤU HÌNH & THƯ VIỆN DÙNG CHUNG**

Notebook này tái lập quy trình (tiền xử lý → FastDTW → 3 đặc trưng) và đã sửa các điểm sau so với bản cũ:
- Không còn chọn ngưỡng/mô hình trên tập test.
- Bỏ cặp video giả (giáo viên so với chính nó), chỉ dùng cặp thật.
- Mỗi khung hình học viên chỉ ghép với **đúng 1** khung hình giáo viên sau DTW (bản cũ có khung bị nhân đôi số dòng, điểm có thể vượt 13).
- Đặt seed, in thông tin phần cứng.

In [ ]:
import os, time, hashlib, platform, warnings
import numpy as np
import pandas as pd
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATA_DIR = '.'                      # thư mục chứa student_dance_*.csv / teacher_dance_*.csv
VIS_THRESHOLD = 0.5                 # Visibility < 0.5 -> coi là mất dấu (NaN)
MOVEMENT_STD_THRESHOLD = 0.015      # lọc video học viên đứng yên

JOINTS = ['nose',
          'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist',
          'left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle']
N_JOINTS = len(JOINTS)              # 13
FEATURES = ['Joint_Error', 'Joint_Cosine', 'Velocity_Error']
LABEL = 'Ground_Truth_Label'

# 13 cặp video thật (bỏ cặp id=13 là giáo viên so với chính nó; id=14 không dùng)
PAIR_IDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15]
TEST_VIDEO_IDS = [5, 9]            # 2 cặp để test, 11 cặp còn lại để train

# True: loại khỏi train các video có video giáo viên trùng với video test (xem cảnh báo ở phần chia dữ liệu)
EXCLUDE_TRAIN_SHARING_TEACHER_WITH_TEST = False

CATBOOST_PARAMS = dict(iterations=150, depth=6, learning_rate=0.08, random_seed=SEED, verbose=0)

print('Phần cứng:', platform.processor() or platform.machine(), '| CPU logic:', os.cpu_count(),
      '|', platform.system(), platform.release(), '| Python', platform.python_version())

**1. TIỀN XỬ LÝ (VISIBILITY → NỘI SUY → BỎ TRỤC Z → GỐC HÔNG → CHIA THEO THÂN NGƯỜI)**

In [ ]:
def is_dancing(df_student, threshold=MOVEMENT_STD_THRESHOLD):
    """Bộ lọc video đứng yên: độ lệch chuẩn trung bình của cổ tay và cổ chân."""
    key_joints = ['left_wrist', 'right_wrist', 'left_ankle', 'right_ankle']
    d = df_student[df_student['Joint_Name'].isin(key_joints)]
    movement_std = d.groupby('Joint_Name')[['X', 'Y']].std().mean().mean()
    return movement_std >= threshold, movement_std


def preprocess_data(df):
    """Trả về dict: frames, X, Y (đã chuẩn hóa, shape [n_frames, 13]) và label (nếu có)."""
    df = df.sort_values(['Frame', 'Joint_Name']).reset_index(drop=True).copy()

    # Mất dấu -> NaN -> nội suy tuyến tính theo từng khớp (chỉ dùng X, Y; bỏ trục Z)
    df.loc[df['Visibility'] < VIS_THRESHOLD, ['X', 'Y']] = np.nan
    df[['X', 'Y']] = df.groupby('Joint_Name')[['X', 'Y']].transform(
        lambda s: s.interpolate(method='linear', limit_direction='both'))
    df[['X', 'Y']] = df[['X', 'Y']].fillna(0)

    X = df.pivot(index='Frame', columns='Joint_Name', values='X')[JOINTS]
    Y = df.pivot(index='Frame', columns='Joint_Name', values='Y')[JOINTS]

    root_x = (X['left_hip'] + X['right_hip']) / 2
    root_y = (Y['left_hip'] + Y['right_hip']) / 2
    sh_x = (X['left_shoulder'] + X['right_shoulder']) / 2
    sh_y = (Y['left_shoulder'] + Y['right_shoulder']) / 2
    torso = np.maximum(np.hypot(sh_x - root_x, sh_y - root_y), 1e-5)

    out = {
        'frames': X.index.to_numpy(),
        'X': X.sub(root_x, axis=0).div(torso, axis=0).to_numpy(),
        'Y': Y.sub(root_y, axis=0).div(torso, axis=0).to_numpy(),
    }
    if 'Label' in df.columns:
        out['label'] = df.pivot(index='Frame', columns='Joint_Name', values='Label')[JOINTS].to_numpy()
    return out

**2. FASTDTW + TRÍCH XUẤT 3 ĐẶC TRƯNG**

Sau DTW, mỗi khung hình học viên được ghép với đúng **một** khung hình giáo viên (khung có khoảng cách nhỏ nhất trong các khung được ghép). Nhờ vậy mỗi khung luôn có đúng 13 dòng và điểm $S_{frame}\in[0,13]$.
Vận tốc được tính trên chuỗi **đã gióng hàng** (sai phân bậc nhất giữa hai khung liên tiếp của chuỗi đã gióng).

In [ ]:
def align_and_extract(seq_s, seq_t):
    vs = np.hstack([seq_s['X'], seq_s['Y']])
    vt = np.hstack([seq_t['X'], seq_t['Y']])
    _, path = fastdtw(vs, vt, dist=euclidean)

    best = {}                                   # khung học viên i -> (khoảng cách, khung giáo viên j)
    for i, j in path:
        d = np.linalg.norm(vs[i] - vt[j])
        if i not in best or d < best[i][0]:
            best[i] = (d, j)
    t_idx = np.array([best[i][1] for i in range(len(vs))])

    Xs, Ys = seq_s['X'], seq_s['Y']
    Xt, Yt = seq_t['X'][t_idx], seq_t['Y'][t_idx]

    def vel(a):                                  # sai phân bậc nhất, khung đầu = 0
        return np.diff(a, axis=0, prepend=a[:1])

    joint_error = np.hypot(Xs - Xt, Ys - Yt)
    joint_cosine = (Xs * Xt + Ys * Yt) / (np.hypot(Xs, Ys) * np.hypot(Xt, Yt) + 1e-9)
    velocity_error = np.hypot(vel(Xs) - vel(Xt), vel(Ys) - vel(Yt))

    n = len(vs)
    return pd.DataFrame({
        'Frame_s': np.repeat(seq_s['frames'], N_JOINTS),
        'Joint_Name': np.tile(JOINTS, n),
        'Joint_Error': joint_error.ravel(),
        'Joint_Cosine': joint_cosine.ravel(),
        'Velocity_Error': velocity_error.ravel(),
        LABEL: seq_s['label'].ravel().astype(int),
    })


def file_md5(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

**3. XÂY DỰNG TẬP DỮ LIỆU TỪ CÁC CẶP VIDEO**

In [ ]:
parts, teacher_group = [], {}
print('=== TRÍCH XUẤT ĐẶC TRƯNG ===')
for vid in PAIR_IDS:
    s_path = os.path.join(DATA_DIR, f'student_dance_{vid}.csv')
    t_path = os.path.join(DATA_DIR, f'teacher_dance_{vid}.csv')
    df_s, df_t = pd.read_csv(s_path), pd.read_csv(t_path)

    moving, std = is_dancing(df_s)
    if not moving:
        print(f'[BỘ LỌC] Bỏ video {vid}: đứng yên (Std={std:.4f})')
        continue

    feats = align_and_extract(preprocess_data(df_s), preprocess_data(df_t))
    feats['Video_ID'] = vid
    parts.append(feats)
    teacher_group[vid] = file_md5(t_path)       # video giáo viên giống hệt nhau -> cùng nhóm
    print(f'[OK] Video {vid}: {feats["Frame_s"].nunique()} khung hình')

master_dataset = pd.concat(parts, ignore_index=True)

assert (master_dataset.groupby(['Video_ID', 'Frame_s']).size() == N_JOINTS).all(), 'mỗi khung phải có đúng 13 dòng'

all_ids = sorted(int(v) for v in master_dataset['Video_ID'].unique())
train_ids = [v for v in all_ids if v not in TEST_VIDEO_IDS]
test_ids = [v for v in all_ids if v in TEST_VIDEO_IDS]

test_teachers = {teacher_group[v] for v in test_ids}
overlap = [v for v in train_ids if teacher_group[v] in test_teachers]
if overlap:
    print(f'\n[CẢNH BÁO] Video train {overlap} dùng cùng video giáo viên với video test {test_ids}.')
    if EXCLUDE_TRAIN_SHARING_TEACHER_WITH_TEST:
        train_ids = [v for v in train_ids if v not in overlap]
        print('  -> đã loại khỏi tập train.')
    else:
        print('  -> vẫn giữ trong train (đặt EXCLUDE_TRAIN_SHARING_TEACHER_WITH_TEST=True để loại).')

train_data = master_dataset[master_dataset['Video_ID'].isin(train_ids)].reset_index(drop=True)
test_data = master_dataset[master_dataset['Video_ID'].isin(test_ids)].reset_index(drop=True)

n_tr = train_data[['Video_ID', 'Frame_s']].drop_duplicates().shape[0]
n_te = test_data[['Video_ID', 'Frame_s']].drop_duplicates().shape[0]
print(f'\nTRAIN: {len(train_ids)} cặp {train_ids} | {n_tr:,} khung hình')
print(f'TEST : {len(test_ids)} cặp {test_ids} | {n_te:,} khung hình')
print('Tỉ lệ nhãn khớp (1=đúng) train/test:',
      round(train_data[LABEL].mean(), 3), '/', round(test_data[LABEL].mean(), 3))

**4. PHÂN LOẠI CẤP KHỚP + CHỌN NGƯỠNG KHUNG HÌNH (KHÔNG RÒ RỈ DỮ LIỆU)**

Quy tắc: tập test **chỉ dùng một lần ở cuối**. Ngưỡng $T$ được chọn bằng *leave-one-group-out cross-validation* trên tập train (nhóm = các video dùng chung một video giáo viên), dựa trên nhãn khớp dự đoán ngoài-fold (out-of-fold).

Vì $S_{frame}$ là tổng của 13 nhãn nhị phân nên chỉ là số nguyên: một khung là PASS nếu $S_{frame}\ge k$ với $k\in\{1,\dots,13\}$ (ngưỡng thực $T=7.1$ ở bản cũ tương đương $k=8$).

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import (accuracy_score, f1_score, recall_score, precision_score,
                             classification_report, confusion_matrix)

MIN_RECALL_FAIL = 0.90
MAJORITY = N_JOINTS // 2 + 1        # khung "đúng" thật nếu >= 7/13 khớp đúng


def fit_catboost(df):
    model = CatBoostClassifier(**CATBOOST_PARAMS)
    model.fit(df[FEATURES], df[LABEL])
    return model


def predict_joint_labels(model, df):
    return np.asarray(model.predict(df[FEATURES])).ravel().astype(int)


def frame_table(df, pred_col='Pred'):
    """Gộp 13 khớp/khung -> (nhãn thật của khung, điểm S_frame)."""
    g = df.groupby(['Video_ID', 'Frame_s'])
    return pd.DataFrame({
        'actual': (g[LABEL].sum() >= MAJORITY).astype(int),
        'score': g[pred_col].sum(),
    }).reset_index()


def sweep_thresholds(frames):
    rows = []
    y = frames['actual'].to_numpy()
    for k in range(1, N_JOINTS + 1):
        p = (frames['score'].to_numpy() >= k).astype(int)
        rows.append({
            'k (S>=k)': k,
            'Acc (%)': 100 * accuracy_score(y, p),
            'Recall_Fail (%)': 100 * recall_score(y, p, pos_label=0, zero_division=0),
            'Recall_Pass (%)': 100 * recall_score(y, p, pos_label=1, zero_division=0),
            'F1-Macro': f1_score(y, p, average='macro', zero_division=0),
        })
    return pd.DataFrame(rows)


def select_threshold(sweep_df, min_recall_fail=MIN_RECALL_FAIL):
    ok = sweep_df[sweep_df['Recall_Fail (%)'] >= 100 * min_recall_fail]
    if ok.empty:
        print(f'[!] Không k nào đạt Recall_Fail >= {min_recall_fail}; chọn k có F1-Macro cao nhất.')
        ok = sweep_df
    return int(ok.loc[ok['F1-Macro'].idxmax(), 'k (S>=k)'])

**4.1 Chọn ngưỡng bằng cross-validation trên tập TRAIN (leave-one-group-out)**

In [ ]:
oof_parts = []
groups = sorted({teacher_group[v] for v in train_ids})
print(f'{len(groups)} fold (mỗi fold bỏ ra các video dùng chung một video giáo viên)')

for gi, g in enumerate(groups, 1):
    val_vids = [v for v in train_ids if teacher_group[v] == g]
    tr = train_data[~train_data['Video_ID'].isin(val_vids)]
    va = train_data[train_data['Video_ID'].isin(val_vids)].copy()
    va['Pred'] = predict_joint_labels(fit_catboost(tr), va)
    oof_parts.append(va)
    print(f'  fold {gi:>2}: bỏ ra video {val_vids}')

oof = pd.concat(oof_parts, ignore_index=True)
oof_frames = frame_table(oof)

cv_sweep = sweep_thresholds(oof_frames)
print('\n=== GRID SEARCH NGƯỠNG TRÊN OUT-OF-FOLD (TRAIN) ===')
print(cv_sweep.round(4).to_string(index=False))

K_SELECTED = select_threshold(cv_sweep)
print(f'\n=> Ngưỡng chọn từ cross-validation: k = {K_SELECTED}  (PASS nếu S_frame >= {K_SELECTED}/13)')

**4.2 Huấn luyện mô hình cuối trên toàn bộ TRAIN, đánh giá TEST đúng một lần với $k$ đã chọn**

In [ ]:
final_model = fit_catboost(train_data)
test_eval = test_data.copy()
test_eval['Pred'] = predict_joint_labels(final_model, test_eval)
test_frames = frame_table(test_eval)

y_true = test_frames['actual'].to_numpy()
y_pred = (test_frames['score'].to_numpy() >= K_SELECTED).astype(int)

print('=' * 70)
print(f'KẾT QUẢ TRÊN TEST (video {test_ids}) | CatBoost | PASS nếu S_frame >= {K_SELECTED}/13')
print('=' * 70)
print(classification_report(y_true, y_pred, target_names=['Fail (0)', 'Pass (1)'], digits=2))
print(f'Accuracy = {100 * accuracy_score(y_true, y_pred):.2f}%')
print(f'Baseline lớp đa số (luôn đoán Pass) = {100 * y_true.mean():.2f}%')

cm = confusion_matrix(y_true, y_pred)
print(f'\nMa trận nhầm lẫn ({len(y_true)} khung): TN={cm[0][0]} FP={cm[0][1]} FN={cm[1][0]} TP={cm[1][1]}')
print(f"Recall_Fail = {100 * cm[0][0] / cm[0].sum():.2f}%  (bỏ sót lỗi: {cm[0][1]}/{cm[0].sum()})")

print('\nPhân bố nhãn thật trên test theo từng video:')
print(test_frames.groupby('Video_ID')['actual'].agg(frames='count', pass_frames='sum').to_string())

**4.3 Phân tích độ nhạy ngưỡng trên TEST (chỉ để mô tả/đối chiếu, KHÔNG dùng để chọn $k$)**

Bảng này tương ứng với bảng grid search cũ trong bài. Khi đưa vào bài phải ghi rõ là phân tích độ nhạy trên tập test, còn ngưỡng đã được chọn từ cross-validation ở 4.1.

In [ ]:
test_sweep = sweep_thresholds(test_frames)
print(test_sweep.round(4).to_string(index=False))

**4.4 Chỉ số cấp khớp của mô hình cuối và độ quan trọng đặc trưng** (để báo cáo cùng mô hình với kết quả cấp khung hình)

In [ ]:
yj, pj = test_eval[LABEL].to_numpy(), test_eval['Pred'].to_numpy()
print(f'Cấp KHỚP trên test: Accuracy = {100 * accuracy_score(yj, pj):.2f}% | Macro-F1 = {f1_score(yj, pj, average="macro"):.4f}')
print(f'Baseline lớp đa số cấp khớp = {100 * max(yj.mean(), 1 - yj.mean()):.2f}%')
imp = final_model.get_feature_importance()
print('Độ quan trọng đặc trưng:', ', '.join(f'{c} ({v:.1f}%)' for c, v in zip(FEATURES, imp)))

**5. KIỂM ĐỊNH NỘI SUY TUYẾN TÍNH (CÓ SEED, LẶP NHIỀU LẦN, TRÊN CHUỖI LIÊN TỤC)**

- Lấy **tất cả** các đoạn 100 khung hình liên tiếp (không chồng lấn) có Visibility > 0.95 của khớp cần kiểm tra, trên toàn bộ video học viên.
- Hai kịch bản mất dữ liệu 15%: (a) 15 khung rời rạc ngẫu nhiên; (b) 3 đoạn mất liên tiếp, mỗi đoạn 5 khung (giống che khuất thật hơn).
- Chỉ xóa các khung bên trong đoạn (không xóa khung đầu/cuối) để không phải ngoại suy.
- So sánh nội suy tuyến tính với spline bậc 3. Mỗi đoạn lặp 10 lần; báo cáo trung bình ± độ lệch chuẩn **giữa các đoạn**.
- Tọa độ là X gốc của MediaPipe (chưa chuẩn hóa theo thân người), giống thí nghiệm cũ.

In [ ]:
INTERP_WINDOW = 100
INTERP_MISSING = 0.15
INTERP_REPEATS = 10
INTERP_CASES = {'right_knee': 'Khớp đầu gối phải', 'left_elbow': 'Khớp khuỷu tay trái'}


def clean_windows(df, joint, length=INTERP_WINDOW, vis=0.95):
    # Các đoạn `length` khung liên tiếp (không chồng lấn) có Visibility > vis; trả về tọa độ X
    d = df[df['Joint_Name'] == joint].sort_values('Frame')
    good = (d['Visibility'] > vis).to_numpy()
    frames = d['Frame'].to_numpy()
    x = d['X'].to_numpy()
    wins, run = [], 0
    for i in range(len(d)):
        if not good[i]:
            run = 0
        elif run > 0 and frames[i] - frames[i - 1] == 1:
            run += 1
        else:
            run = 1
        if run >= length:
            wins.append(x[i - length + 1:i + 1])
            run = 0
    return wins


def make_missing_idx(rng, n, ratio, scenario):
    k = int(round(n * ratio))
    if scenario == 'random':
        return rng.choice(np.arange(1, n - 1), size=k, replace=False)
    seg = k // 3                                  # 'block': 3 đoạn liên tiếp, không chồng lấn
    while True:
        starts = np.sort(rng.integers(1, n - 1 - seg, size=3))
        if np.all(np.diff(starts) >= seg + 1):
            return np.concatenate([np.arange(s, s + seg) for s in starts])


def run_interp(series, idx, method):
    s = pd.Series(series.copy())
    s.iloc[idx] = np.nan
    return s.interpolate(method=method).to_numpy()


windows = {j: [] for j in INTERP_CASES}
for vid in PAIR_IDS:
    df_v = pd.read_csv(os.path.join(DATA_DIR, f'student_dance_{vid}.csv'))
    for j in INTERP_CASES:
        windows[j] += clean_windows(df_v, j)
print({INTERP_CASES[j]: f'{len(w)} đoạn' for j, w in windows.items()})

rows = []
for joint, name in INTERP_CASES.items():
    for scenario in ['random', 'block']:
        for method in ['linear', 'cubic']:
            rng = np.random.default_rng(SEED)
            rmse_w, mae_w = [], []
            for series in windows[joint]:
                r, m = [], []
                for _ in range(INTERP_REPEATS):
                    idx = make_missing_idx(rng, len(series), INTERP_MISSING, scenario)
                    err = series[idx] - run_interp(series, idx, method)[idx]
                    r.append(np.sqrt(np.mean(err ** 2)))
                    m.append(np.mean(np.abs(err)))
                rmse_w.append(np.mean(r)); mae_w.append(np.mean(m))
            rows.append({'Khớp': name, 'Kịch bản': scenario, 'Phương pháp': method, 'Số đoạn': len(rmse_w),
                         'RMSE': f'{np.mean(rmse_w):.4f} ± {np.std(rmse_w):.4f}',
                         'MAE': f'{np.mean(mae_w):.4f} ± {np.std(mae_w):.4f}'})
print(pd.DataFrame(rows).to_string(index=False))

**5.1 Xuất lại hình minh họa nội suy** — ghi đè `interp_knee.png` và `interp_elbow.png`.
Hình minh họa dùng đoạn có biến thiên lớn nhất (độ lệch chuẩn lớn nhất) của mỗi khớp, một lần chạy với seed cố định; phải ghi rõ trong chú thích hình.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

for joint, name, out in [('right_knee', 'Khớp đầu gối phải', 'interp_knee.png'),
                         ('left_elbow', 'Khớp khuỷu tay trái', 'interp_elbow.png')]:
    series = max(windows[joint], key=np.std)
    rng = np.random.default_rng(SEED)
    idx = make_missing_idx(rng, len(series), INTERP_MISSING, 'random')
    rec = run_interp(series, idx, 'linear')
    rmse = np.sqrt(np.mean((series[idx] - rec[idx]) ** 2))
    mae = np.mean(np.abs(series[idx] - rec[idx]))
    print(f'{name}: RMSE = {rmse:.4f}, MAE = {mae:.4f} (đoạn biến thiên lớn nhất, seed {SEED}, 1 lần chạy)')

    plt.figure(figsize=(12, 5))
    plt.plot(series, label='Dữ liệu gốc', color='blue', alpha=0.5, linewidth=2)
    plt.plot(rec, label='Nội suy tuyến tính', color='red', linestyle='dashed')
    plt.scatter(idx, rec[idx], color='orange', label='Các khung được nội suy', zorder=5)
    plt.xlabel('Chỉ số khung hình'); plt.ylabel('Tọa độ X (chuẩn hóa theo ảnh)')
    plt.title(name); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, out), dpi=200); plt.close()